In [47]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage, HumanMessage, AIMessage # response from a tool will be a tool Message


In [48]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [49]:
multiply.invoke({"a": 1, "b": 2})

2

In [50]:
llm = ChatOpenAI()

In [51]:
llm_with_tools = llm.bind_tools([multiply])

In [52]:
llm_with_tools.invoke('Hi how are you') # no tool call involved here.

AIMessage(content="Hello! I'm here and ready to help you with any questions or tasks you have. What can I assist you with today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 58, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EZB6fRIjDeCl4xQBZu8sQ6GS3f371', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13eac-e6fa-7532-a98e-b12bd82adfe5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 27, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [53]:
# making llm to call the tool we built

messages = [];

firstMessage = HumanMessage("multiple 2 with 6")
messages.append(firstMessage)
initialResponse: AIMessage = llm_with_tools.invoke(messages)
messages.append(initialResponse)

print(initialResponse.tool_calls) # AI message has tool_calls property
# the reponse has a property called tool_calls, which contains all the necessary payload e.t.c that are require to invoke a tool

""" Smaple tool_calls
 tool_calls=[{'name': 'multiply', 'args': {'a': 1, 'b': 6}, 'id': 'call_tNvpOxJPLOSiFCTP3E8WPQRx', 'type': 'tool_call'}]
"""

[{'name': 'multiply', 'args': {'a': 2, 'b': 6}, 'id': 'call_XiZtHp72gsiiXAZXzcYH6czA', 'type': 'tool_call'}]


" Smaple tool_calls\n tool_calls=[{'name': 'multiply', 'args': {'a': 1, 'b': 6}, 'id': 'call_tNvpOxJPLOSiFCTP3E8WPQRx', 'type': 'tool_call'}]\n"

In [54]:
# Now i have to use this message and need to invoke the custom tool we created.

toolMessage = initialResponse.tool_calls[0]
print(toolMessage)

{'name': 'multiply', 'args': {'a': 2, 'b': 6}, 'id': 'call_XiZtHp72gsiiXAZXzcYH6czA', 'type': 'tool_call'}


In [59]:
# invoking the tool


toolResponse = multiply.invoke(toolMessage) # this tool takes ToolCall as input argument and returns ToolMessage
messages.append(toolResponse)
print(toolResponse)
print("messages ", messages)

content='12' name='multiply' tool_call_id='call_XiZtHp72gsiiXAZXzcYH6czA'
messages  [HumanMessage(content='multiple 2 with 6', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 60, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EZB6hZouLqSqiJLGi22zeiMOLjlCG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13eac-eefe-71c3-b8d6-b2756ae7e740-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 6}, 'id': 'call_XiZtHp72gsiiXAZXzcYH6czA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_toke

In [60]:
# sending the tool resonse back to llm
print(messages)
finalResponse = llm_with_tools.invoke(messages)
print(finalResponse.content)

[HumanMessage(content='multiple 2 with 6', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 60, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EZB6hZouLqSqiJLGi22zeiMOLjlCG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13eac-eefe-71c3-b8d6-b2756ae7e740-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 6}, 'id': 'call_XiZtHp72gsiiXAZXzcYH6czA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 17, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, '

BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[3].role', 'code': None}}